In [1]:
import pandas as pd

# Preguntas de análisis usando los archivos .parquet

Leer y almacenar el contenido de los archivos

In [2]:
data_categories = pd.read_parquet('categorias.parquet')

data_books = pd.read_parquet('libros.parquet')

In [3]:
data_categories.head()

,categoria,url_categoria,cantidad_libros,fecha_extraccion,extraido_por
0,Travel,https://books.toscrape.com/catalogue/category/...,11,2026-08-08 12:01:33.401756,Miguel Zuleta
1,Mystery,https://books.toscrape.com/catalogue/category/...,32,2026-08-08 12:01:34.064678,Miguel Zuleta
2,Historical Fiction,https://books.toscrape.com/catalogue/category/...,26,2026-08-08 12:01:34.841400,Miguel Zuleta
3,Sequential Art,https://books.toscrape.com/catalogue/category/...,75,2026-08-08 12:01:35.530058,Miguel Zuleta
4,Classics,https://books.toscrape.com/catalogue/category/...,19,2026-08-08 12:01:36.110119,Miguel Zuleta


In [4]:
data_books.tail()

,upc,titulo,categoria,descripcion,tipo_producto,precio_sin_impuesto,precio_con_impuesto,impuesto,moneda,disponibilidad,cantidad_stock,calificacion,cantidad_resenas,url_libro,url_imagen,fecha_extraccion,extraido_por
995,cd2a2a70dd5d176d,Alice in Wonderland (Alice's Adventures in Won...,Classics,,Books,55.53,55.53,0.0,GBP,In stock (1 available),1,1,0,https://books.toscrape.com/catalogue/alice-in-...,https://books.toscrape.com/media/cache/99/df/9...,2026-08-08 12:11:04.083679,Miguel Zuleta
996,bfd5e1701c862ac3,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",Sequential Art,High school student Kei Nagai is struck dead i...,Books,57.06,57.06,0.0,GBP,In stock (1 available),1,4,0,https://books.toscrape.com/catalogue/ajin-demi...,https://books.toscrape.com/media/cache/30/98/3...,2026-08-08 12:11:04.529073,Miguel Zuleta
997,19fec36a1dfb4c16,A Spy's Devotion (The Regency Spies of London #1),Historical Fiction,"In England’s Regency era, manners and elegance...",Books,16.97,16.97,0.0,GBP,In stock (1 available),1,5,0,https://books.toscrape.com/catalogue/a-spys-de...,https://books.toscrape.com/media/cache/f9/6b/f...,2026-08-08 12:11:04.972107,Miguel Zuleta
998,f684a82adc49f011,1st to Die (Women's Murder Club #1),Mystery,"James Patterson, bestselling author of the Ale...",Books,53.98,53.98,0.0,GBP,In stock (1 available),1,1,0,https://books.toscrape.com/catalogue/1st-to-di...,https://books.toscrape.com/media/cache/f6/8e/f...,2026-08-08 12:11:05.259886,Miguel Zuleta
999,228ba5e7577e1d49,"1,000 Places to See Before You Die",Travel,"Around the World, continent by continent, here...",Books,26.08,26.08,0.0,GBP,In stock (1 available),1,5,0,https://books.toscrape.com/catalogue/1000-plac...,https://books.toscrape.com/media/cache/9e/10/9...,2026-08-08 12:11:05.685220,Miguel Zuleta


## ¿Cuántas categorías de libros existen?

In [17]:
cantidad_categorias = len(data_categories)
print(cantidad_categorias)

50


Existen 50 categorías de libros

## ¿Cuántos libros hay en cada categoría?

In [21]:
data_categories[['categoria', 'cantidad_libros']]

,categoria,cantidad_libros
0,Travel,11
1,Mystery,32
2,Historical Fiction,26
3,Sequential Art,75
4,Classics,19
5,Philosophy,11
6,Romance,35
7,Womens Fiction,17
8,Fiction,65
9,Childrens,29


## ¿Cuál es el libro más caro? 

In [20]:
precio_maximo = data_books["precio_con_impuesto"].max()

libros_mas_caros = data_books[data_books["precio_con_impuesto"] == precio_maximo]

libros_mas_caros[["titulo", "precio_con_impuesto"]]

,titulo,precio_con_impuesto
648,The Perfect Play (Play by Play #1),59.99


The Perfect Play (Play by Play #1) es el libro más caro, con un precio de £59.59

## ¿Hay algún libro que aparezca en más de una categoría?

In [23]:
duplicados = data_books[data_books["upc"].duplicated(keep=False)]

duplicados[["upc", "titulo", "categoria"]].sort_values("upc")

,upc,titulo,categoria


In [25]:
cantidad = data_books[data_books["upc"].duplicated()]["upc"].nunique()

print(cantidad)

0


Esto solo significa que no hay UPC repetidos (lo cual, para el dataset actual. significa que no hay libros que pertenezcan a más de una categoría), pero es el único análisis posible con respecto al upc, a pesar de que puede no ser muy informativo considerando el hecho de que el proceso de extracción solo conserva una fila por UPC.

## ¿Cuál es el libro más barato de cada categoría?

In [28]:
precios_minimos = data_books.groupby("categoria")["precio_con_impuesto"].transform("min")

libros_mas_baratos = data_books[
    data_books["precio_con_impuesto"] == precios_minimos
]

libros_mas_baratos[[
    "categoria",
    "titulo",
    "precio_con_impuesto"
]].sort_values("categoria")

,categoria,titulo,precio_con_impuesto
616,Academic,Logan Kade (Fallen Crest High #5.5),13.12
716,Add a comment,The Tipping Point: How Little Things Can Make ...,10.02
844,Adult Fiction,Fifty Shades Freed (Fifty Shades #3),15.36
479,Art,History of Beauty,10.29
163,Autobiography,The Argonauts,10.93
182,Biography,Louisa: The Extraordinary Life of Mrs. Adams,16.85
138,Business,The Third Wave: An Entrepreneur’s Vision of th...,12.61
858,Childrens,Counting Thyme,10.62
539,Christian,Blue Like Jazz: Nonreligious Thoughts on Chris...,25.77
537,Christian Fiction,Counted With the Stars (Out from Egypt #1),17.97


## ¿Cuánto más caro o más barato es cada libro respecto al precio promedio de su categoría?

In [30]:
promedio_categoria = data_books.groupby("categoria")["precio_con_impuesto"].transform("mean")

data_books["diferencia_precio"] = (
    data_books["precio_con_impuesto"] - promedio_categoria
)

data_books[[
    "categoria",
    "titulo",
    "precio_con_impuesto",
    "diferencia_precio"
]]

,categoria,titulo,precio_con_impuesto,diferencia_precio
0,Poetry,A Light in the Attic,51.77,15.795789
1,Historical Fiction,Tipping the Velvet,53.74,20.095769
2,Fiction,Soumission,50.10,14.033385
3,Mystery,Sharp Objects,47.82,16.100938
4,History,Sapiens: A Brief History of Humankind,54.23,16.935000
...,...,...,...,...
995,Classics,Alice in Wonderland (Alice's Adventures in Won...,55.53,18.984737
996,Sequential Art,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",57.06,22.487733
997,Historical Fiction,A Spy's Devotion (The Regency Spies of London #1),16.97,-16.674231
998,Mystery,1st to Die (Women's Murder Club #1),53.98,22.260937


## Mayor ingreso potencial por categoría

In [36]:
data_books["ingreso_potencial"] = (
    data_books["precio_con_impuesto"] * data_books["cantidad_stock"]
)

ingreso_maximo = data_books.groupby("categoria")["ingreso_potencial"].transform("max")

libros_mayor_ingreso = data_books[
    data_books["ingreso_potencial"] == ingreso_maximo
]

libros_mayor_ingreso[[
    "categoria",
    "titulo",
    "precio_con_impuesto",
    "cantidad_stock",
    "ingreso_potencial"
]].sort_values("categoria")

,categoria,titulo,precio_con_impuesto,cantidad_stock,ingreso_potencial
616,Academic,Logan Kade (Fallen Crest High #5.5),13.12,5,65.60
97,Add a comment,Judo: Seven Steps to Black Belt (an Introducto...,53.90,16,862.40
844,Adult Fiction,Fifty Shades Freed (Fifty Shades #3),15.36,3,46.08
29,Art,Wall and Piece,44.18,18,795.24
405,Autobiography,Lab Girl,40.85,11,449.35
540,Biography,Benjamin Franklin: An American Life,48.19,7,337.33
6,Business,The Dirty Little Secrets of Getting Your Dream...,33.34,19,633.46
25,Childrens,Birdsong: A Story in Pictures,54.64,19,1038.16
127,Christian,(Un)Qualified: How God Uses Broken People to D...,54.00,16,864.00
202,Christian Fiction,Close to You,49.46,15,741.90
